We need ColumnTransformer to simplify data preprocessing for datasets containing heterogeneous data types (e.g., mixed numerical and categorical features) by allowing different transformations to be applied to specific columns simultaneously.

Without it, preprocessing requires manually extracting, transforming, and concatenating individual columns, which is laborious, error-prone, and difficult to maintain

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [ ]:
df = pd.read_csv('covid_toy.csv')

In [ ]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [ ]:
df.isnull().sum()

,0
age,0
gender,0
fever,10
cough,0
city,0
has_covid,0


From the data we can see that for gender and city we will apply one hot encoding, for cough we apply ordinal encding. Also since there are missing values in the fever coloumn we would have to impute those values. Let's see how we can do that in two different ways.

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['has_covid']),df['has_covid'], test_size=0.2)

The first way is what I call brute force method. Let's see how this works.

First we Impute the missing values.

In [ ]:
# adding simple imputer to fever col
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])

# also the test data
X_test_fever = si.fit_transform(X_test[['fever']])

X_train_fever.shape

(80, 1)

Ordinal encoding the ordinal categorical coloumn of the data. (cough)

In [ ]:
# Ordinalencoding -> cough
oe = OrdinalEncoder(categories=[['Mild','Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])

# also the test data
X_test_cough = oe.fit_transform(X_test[['cough']])

X_train_cough.shape

(80, 1)

One hot encoding the nominal categorical data. (gender and city)

In [ ]:
# OneHotEncoding -> gender,city
ohe = OneHotEncoder(drop='first',sparse_output=False)
X_train_gender_city = ohe.fit_transform(X_train[['gender','city']])

# also the test data
X_test_gender_city = ohe.fit_transform(X_test[['gender','city']])

X_train_gender_city.shape

(80, 4)

Extracting the age coloumn separately.

In [ ]:
# Extracting Age
X_train_age = X_train.drop(columns=['gender','fever','cough','city']).values

# also the test data
X_test_age = X_test.drop(columns=['gender','fever','cough','city']).values

X_train_age.shape

(80, 1)

Concatinating all the different numpy arrays that we got.

In [ ]:
X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)
# also the test data
X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough),axis=1)

X_train_transformed.shape

(80, 7)

And now we complete the encoding part for the data. Pretty time consuming and hectic as you can see.

Now we will see how all this can be done simply using column transformer.

In [ ]:
from sklearn.compose import ColumnTransformer # importing the column transformer class


transformer = ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tnf3',OneHotEncoder(sparse_output=False,drop='first'),['gender','city'])
],remainder='passthrough')

ColumnTransformer accepts a list of column-wise transformations (imputation, encoding, scaling, etc.), applies each to the specified columns, and combines the results into a single transformed NumPy array ready for machine learning.

In [ ]:
transformer.fit_transform(X_train).shape

(80, 7)

In [ ]:
transformer.transform(X_test).shape

(20, 7)